In [2]:

import pandas as pd

# 1. LOAD DATASET
df = pd.read_csv("india_supply_chain_inventory_dirty.csv")


# STEP 1: DATA QUALITY INSPECTION

# 1. Dataset overview
print("Dataset Shape:", df.shape)

# 2. Data types
print("\nData Types:")
print(df.dtypes)

# 3. Missing values
print("\nMissing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 4. Exact duplicates
print("\nExact Duplicate Rows:", df.duplicated().sum())

# 5. Duplicate order IDs
print("\nDuplicate Order IDs:", df["order_id"].duplicated().sum())

# 6. Text format issues
print("\nText Format Issues:")
for col in df.select_dtypes(include=["object", "str"]).columns:
    values = df[col].dropna().astype(str)
    if values.str.strip().ne(values).any():
        print(f"{col}: Extra spaces found")

# 7. Possible missing-value placeholders
print("\nMissing-Value Placeholders:")
for col in df.select_dtypes(include=["object", "str"]).columns:
    count = df[col].astype("string").str.strip().str.lower().isin(
        ["", "na", "n/a", "null", "none", "-"]
    ).sum()
    if count > 0:
        print(f"{col}: {count}")

# 8. Date format inspection
print("\nDate Samples:")
for col in ["order_date", "promised_date", "received_date"]:
    if col in df.columns:
        print(f"{col}:", df[col].dropna().astype(str).head(10).tolist())

# 9. Numeric format inspection
print("\nNumeric Column Data Types and Samples:")
for col in [
    "order_qty", "received_qty", "unit_cost_inr",
    "gst_rate_pct", "on_hand_stock", "freight_cost_inr"
]:
    if col in df.columns:
        print(col, "|", df[col].dtype, "|",
              df[col].dropna().astype(str).head(5).tolist())

# 10. Unique values to identify inconsistent categories
print("\nCategory and Status Formats:")
for col in ["category", "supplier_city", "warehouse_state", "status"]:
    if col in df.columns:
        print(f"{col}:", df[col].dropna().unique()[:20])

print("\nSTEP 1 COMPLETE — INSPECTION ONLY")

Dataset Shape: (102300, 26)

Data Types:
order_id              str
sku                   str
product_name          str
category              str
hsn_code              str
supplier              str
supplier_gstin        str
supplier_city         str
warehouse             str
warehouse_state       str
order_date            str
promised_date         str
received_date         str
order_qty             str
received_qty          str
unit_cost_inr         str
gst_rate_pct          str
on_hand_stock         str
reorder_point       int64
safety_stock        int64
avg_daily_demand      str
freight_cost_inr      str
transporter           str
payment_terms         str
status                str
notes                 str
dtype: object

Missing Values:
category             1671
hsn_code             3362
supplier             1664
supplier_gstin       4754
warehouse            1241
warehouse_state      1584
order_date            127
received_date       13541
received_qty         2511
unit_cost_inr     

In [3]:
# 1. Missing values and placeholders
print("\n1. MISSING VALUES AND PLACEHOLDERS")

missing_tokens = ["", "NA", "N/A", "null", "none", "-"]

for col in df.columns:
    actual_missing = df[col].isna().sum()

    if pd.api.types.is_string_dtype(df[col].dtype):
        text = df[col].str.strip().str.lower()
        placeholders = text.isin(missing_tokens).sum()
    else:
        placeholders = 0

    if actual_missing or placeholders:
        print(
            f"{col}: Nulls={actual_missing}, "
            f"Placeholders={placeholders}"
        )


1. MISSING VALUES AND PLACEHOLDERS
category: Nulls=1671, Placeholders=422
hsn_code: Nulls=3362, Placeholders=847
supplier: Nulls=1664, Placeholders=443
supplier_gstin: Nulls=4754, Placeholders=1191
warehouse: Nulls=1241, Placeholders=301
warehouse_state: Nulls=1584, Placeholders=426
order_date: Nulls=127, Placeholders=0
received_date: Nulls=13541, Placeholders=0
received_qty: Nulls=2511, Placeholders=615
unit_cost_inr: Nulls=2374, Placeholders=634
on_hand_stock: Nulls=3183, Placeholders=796
avg_daily_demand: Nulls=3209, Placeholders=845
transporter: Nulls=4028, Placeholders=1024
payment_terms: Nulls=3118, Placeholders=766
notes: Nulls=46084, Placeholders=0


In [4]:

import pandas as pd

# ============================================================
# STEP 2: DETAILED DATA QUALITY CHECK
# Inspection only — no data cleaning
# ============================================================

# 1. Missing values and placeholders
print("\n1. MISSING VALUES AND PLACEHOLDERS")

missing_tokens = ["", "NA", "N/A", "null", "none", "-"]

for col in df.columns:
    actual_missing = df[col].isna().sum()

    if pd.api.types.is_string_dtype(df[col].dtype):
        text = df[col].str.strip().str.lower()
        placeholders = text.isin(missing_tokens).sum()
    else:
        placeholders = 0

    if actual_missing or placeholders:
        print(
            f"{col}: Nulls={actual_missing}, "
            f"Placeholders={placeholders}"
        )


# 2. Duplicate records
print("\n2. DUPLICATE RECORDS")

print("Exact duplicate rows:", df.duplicated().sum())

print("Repeated order IDs:", df["order_id"].duplicated().sum())

print("Rows belonging to repeated order IDs:",
      df[df["order_id"].duplicated(keep=False)].shape[0])


# 3. Numeric columns containing non-numeric values
print("\n3. NUMERIC FORMAT ISSUES")

numeric_cols = [
    "order_qty", "received_qty", "unit_cost_inr",
    "gst_rate_pct", "on_hand_stock", "avg_daily_demand",
    "freight_cost_inr", "reorder_point", "safety_stock"
]

for col in numeric_cols:
    if col in df.columns:
        values = df[col].astype("string").str.strip()
        parsed = pd.to_numeric(values, errors="coerce")

        invalid = values.notna() & values.ne("") & parsed.isna()

        print(f"\n{col}:")
        print("Non-numeric values:", invalid.sum())
        print("Examples:", values[invalid].drop_duplicates().head(10).tolist())


# 4. Date format and invalid-date inspection
print("\n4. DATE ISSUES")

date_cols = ["order_date", "promised_date", "received_date"]

for col in date_cols:
    if col in df.columns:
        values = df[col].astype("string").str.strip()

        # Test each common format without changing the original column
        formats = [
            "%Y-%m-%d",
            "%d/%m/%Y",
            "%m/%d/%Y",
            "%d.%m.%Y",
            "%d-%m-%Y",
            "%d-%b-%y",
            "%d-%B-%y"
        ]

        valid_any = pd.Series(False, index=df.index)

        for fmt in formats:
            valid_any |= pd.to_datetime(
                values, format=fmt, errors="coerce"
            ).notna()

        nonempty = values.notna() & values.ne("")
        invalid = nonempty & ~valid_any

        print(f"\n{col}:")
        print("Non-empty values:", nonempty.sum())
        print("Unrecognized/invalid values:", invalid.sum())
        print("Examples:", values[invalid].drop_duplicates().head(10).tolist())


# 5. Inconsistent category, city, state and status values
print("\n5. CATEGORY AND TEXT INCONSISTENCIES")

text_cols = [
    "category", "supplier_city", "warehouse_state",
    "status", "supplier", "sku"
]

for col in text_cols:
    if col in df.columns:
        values = df[col].dropna().astype("string")
        normalized = values.str.strip().str.lower()

        print(f"\n{col}:")
        print("Distinct original values:", values.nunique())
        print("Distinct after case/space normalization:",
              normalized.nunique())
        print("Most common values:")
        print(values.value_counts().head(10).to_dict())


# 6. Potentially invalid numeric values
print("\n6. NUMERIC VALUE RANGE INSPECTION")

for col in numeric_cols:
    if col in df.columns:
        values = pd.to_numeric(df[col], errors="coerce")
        print(f"\n{col}:")
        print("Minimum:", values.min())
        print("Maximum:", values.max())
        print("Negative values:", values.lt(0).sum())


# 7. GST rate values
print("\n7. GST RATE DISTRIBUTION")

gst = pd.to_numeric(df["gst_rate_pct"], errors="coerce")
print(gst.value_counts(dropna=False).sort_index())


# 8. Date logic checks
print("\n8. DATE LOGIC CHECKS")

dates = {
    col: pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    for col in date_cols
}

if "order_date" in dates and "received_date" in dates:
    print(
        "Received before order:",
        (dates["received_date"] < dates["order_date"]).sum()
    )

if "order_date" in dates and "promised_date" in dates:
    print(
        "Promise date before order:",
        (dates["promised_date"] < dates["order_date"]).sum()
    )

print("\nSTEP 2 COMPLETE — INSPECTION ONLY")



1. MISSING VALUES AND PLACEHOLDERS
category: Nulls=1671, Placeholders=422
hsn_code: Nulls=3362, Placeholders=847
supplier: Nulls=1664, Placeholders=443
supplier_gstin: Nulls=4754, Placeholders=1191
warehouse: Nulls=1241, Placeholders=301
warehouse_state: Nulls=1584, Placeholders=426
order_date: Nulls=127, Placeholders=0
received_date: Nulls=13541, Placeholders=0
received_qty: Nulls=2511, Placeholders=615
unit_cost_inr: Nulls=2374, Placeholders=634
on_hand_stock: Nulls=3183, Placeholders=796
avg_daily_demand: Nulls=3209, Placeholders=845
transporter: Nulls=4028, Placeholders=1024
payment_terms: Nulls=3118, Placeholders=766
notes: Nulls=46084, Placeholders=0

2. DUPLICATE RECORDS
Exact duplicate rows: 355
Repeated order IDs: 1501
Rows belonging to repeated order IDs: 3002

3. NUMERIC FORMAT ISSUES

order_qty:
Non-numeric values: 5182
Examples: ['479 PCS', '126 nos', '435 nos', '48 pcs', '506 nos', '197 units', '193 units', '1722 units', '138 units', '323 units']

received_qty:
Non-numer

C:\Users\SUTHAR SHIV\AppData\Local\Temp\ipykernel_4464\3729568160.py:141: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  col: pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\SUTHAR SHIV\AppData\Local\Temp\ipykernel_4464\3729568160.py:141: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  col: pd.to_datetime(df[col], errors="coerce", dayfirst=True)


Received before order: 0
Promise date before order: 0

STEP 2 COMPLETE — INSPECTION ONLY


In [5]:

import pandas as pd

# STEP 3: STANDARDIZE MISSING VALUES
# Work on a copy so the original df remains unchanged

df_clean = df.copy()

# Columns where placeholder strings should represent missing data
placeholder_cols = [
    "category",
    "hsn_code",
    "supplier",
    "supplier_gstin",
    "warehouse",
    "warehouse_state",
    "received_qty",
    "unit_cost_inr",
    "on_hand_stock",
    "avg_daily_demand",
    "transporter",
    "payment_terms"
]

missing_tokens = ["", "NA", "N/A", "null", "none", "-"]

for col in placeholder_cols:
    if col in df_clean.columns:
        values = df_clean[col].astype("string").str.strip()
        mask = values.str.casefold().isin(
            [x.casefold() for x in missing_tokens]
        )
        df_clean[col] = df_clean[col].mask(mask, pd.NA)

# Report missing values after standardization
print("Missing values after placeholder standardization:")

missing_report = df_clean.isna().sum()
missing_report = missing_report[missing_report > 0]

print(missing_report.sort_values(ascending=False))

print("\nDataset shape:", df_clean.shape)
print("\nOriginal dataset remains unchanged:", df.shape)


Missing values after placeholder standardization:
notes               46084
received_date       13541
supplier_gstin       5968
transporter          5078
hsn_code             4209
avg_daily_demand     4054
on_hand_stock        3979
payment_terms        3919
received_qty         3126
unit_cost_inr        3008
supplier             2140
category             2102
warehouse_state      2030
warehouse            1560
order_date            127
dtype: int64

Dataset shape: (102300, 26)

Original dataset remains unchanged: (102300, 26)


In [6]:

# STEP 4: STANDARDIZE TEXT FORMATTING
# Continue using df_clean from Step 3

text_cols = [
    "sku",
    "product_name",
    "category",
    "supplier",
    "supplier_city",
    "warehouse",
    "warehouse_state",
    "transporter",
    "payment_terms",
    "status"
]

# Remove leading and trailing spaces
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.strip()

# Standardize status capitalization and separators
if "status" in df_clean.columns:
    df_clean["status"] = (
        df_clean["status"]
        .str.lower()
        .str.replace("_", " ", regex=False)
        .str.title()
    )

# Standardize category capitalization
if "category" in df_clean.columns:
    df_clean["category"] = df_clean["category"].str.title()

print("Text standardization completed.")

for col in ["category", "supplier_city", "warehouse_state", "status"]:
    if col in df_clean.columns:
        print(f"\n{col}:")
        print(df_clean[col].value_counts(dropna=False).head(25))

print("\nDataset shape:", df_clean.shape)


Text standardization completed.

category:
category
Industrial           12602
Automotive           12543
Electronics          12531
Apparel              12476
Agri Inputs          12448
Fmcg Grocery         12273
Pharma & Health      12201
Home & Kitchen       12176
NaN                   2102
Pharma And Health      323
Grocery                318
Home And Kitchen       299
Nan                      8
Name: count, dtype: int64

supplier_city:
supplier_city
Delhi         8284
Mumbai        8165
Kanpur        6049
Pune          5965
Coimbatore    5939
Ahmedabad     5912
Surat         5859
Ludhiana      5818
Jaipur        5816
Hyderabad     5768
Kochi         5577
Gurugram      5500
Kolkata       5497
Bengaluru     5352
delhi          717
Bombay         692
MUMBAI         687
DELHI          672
New Delhi      652
mumbai         639
ahmedabad      496
kochi          485
kolkata        484
JAIPUR         480
KOCHI          477
Name: count, dtype: int64

warehouse_state:
warehouse_state
Mahara

In [7]:

import pandas as pd

# STEP 5: CONVERT NUMERIC COLUMNS
# Work on a copy so failed conversions can be inspected

df_numeric = df_clean.copy()

# Columns with units, currency symbols, commas, or percentages
numeric_cols = [
    "order_qty",
    "received_qty",
    "unit_cost_inr",
    "gst_rate_pct",
    "on_hand_stock",
    "avg_daily_demand",
    "freight_cost_inr"
]

for col in numeric_cols:
    if col not in df_numeric.columns:
        continue

    original = df_numeric[col].astype("string").str.strip()

    # Remove common currency symbols, labels, units and separators
    cleaned = (
        original
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.replace(r"(?i)\bINR\b", "", regex=True)
        .str.replace(r"(?i)\bRs\.?", "", regex=True)
        .str.replace(r"(?i)\b(PCS|NOS|UNITS)\b", "", regex=True)
        .str.replace(r"/-$", "", regex=True)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    converted = pd.to_numeric(cleaned, errors="coerce")

    # Report non-empty values that failed conversion
    failed = original.notna() & original.ne("") & converted.isna()

    print(f"\n{col}")
    print("Failed conversions:", failed.sum())
    print("Failed examples:", original[failed].drop_duplicates().head(10).tolist())

    df_numeric[col] = converted

# Report resulting data types
print("\nNumeric conversion results:")
print(df_numeric[numeric_cols].dtypes)

print("\nDataset shape:", df_numeric.shape)



order_qty
Failed conversions: 0
Failed examples: []

received_qty
Failed conversions: 0
Failed examples: []

unit_cost_inr
Failed conversions: 0
Failed examples: []

gst_rate_pct
Failed conversions: 0
Failed examples: []

on_hand_stock
Failed conversions: 0
Failed examples: []

avg_daily_demand
Failed conversions: 0
Failed examples: []

freight_cost_inr
Failed conversions: 0
Failed examples: []

Numeric conversion results:
order_qty             Int64
received_qty          Int64
unit_cost_inr       Float64
gst_rate_pct          Int64
on_hand_stock         Int64
avg_daily_demand    Float64
freight_cost_inr    Float64
dtype: object

Dataset shape: (102300, 26)


In [8]:
numeric_cols = [
    "order_qty",
    "received_qty",
    "unit_cost_inr",
    "gst_rate_pct",
    "on_hand_stock",
    "reorder_point",
    "safety_stock",
    "avg_daily_demand",
    "freight_cost_inr"
]

# 1. Check negative values
print("NEGATIVE VALUES")
for col in numeric_cols:
    count = (df_numeric[col] < 0).sum()
    print(f"{col}: {count}")

# 2. Check invalid GST rates
valid_gst_rates = [0, 5, 12, 18, 28]

print("\nINVALID GST RATES")
invalid_gst = ~df_numeric["gst_rate_pct"].isin(valid_gst_rates)
print("Count:", invalid_gst.sum())
print(df_numeric.loc[invalid_gst, "gst_rate_pct"].value_counts(dropna=False))

# 3. Check received quantity greater than ordered quantity
print("\nRECEIVED QTY GREATER THAN ORDER QTY")
condition = (
    df_numeric["received_qty"].notna()
    & df_numeric["order_qty"].notna()
    & (df_numeric["received_qty"] > df_numeric["order_qty"])
)

print("Count:", condition.sum())
print(
    df_numeric.loc[
        condition,
        ["order_id", "sku", "order_qty", "received_qty", "status"]
    ].head(10)
)

# 4. Check zero quantities and costs
print("\nZERO VALUES")
for col in ["order_qty", "received_qty", "unit_cost_inr", "avg_daily_demand"]:
    print(f"{col}: {(df_numeric[col] == 0).sum()}")

NEGATIVE VALUES
order_qty: 622
received_qty: 0
unit_cost_inr: 177
gst_rate_pct: 0
on_hand_stock: 523
reorder_point: 0
safety_stock: 507
avg_daily_demand: 357
freight_cost_inr: 215

INVALID GST RATES
Count: 1647
gst_rate_pct
22     439
3      407
180    402
15     399
Name: count, dtype: Int64

RECEIVED QTY GREATER THAN ORDER QTY
Count: 2838
           order_id        sku  order_qty  received_qty       status
31   PO/2024/017752  SKU-10278       -100            39    Delivered
44   PO/2026/081063  SKU-10096          5            12    Cancelled
82   PO/2023/049899  SKU-10407         76           180    Delivered
120  PO/2026/075231  SKU-10464         46           129  Backordered
132  PO/2024/092142  SKU-10308        113           303    Delivered
141  PO/2023/089274  SKU-10705         76           216    Delivered
250  PO/2026/025158  SKU-10149         -5            93    Cancelled
264  PO/2023/046375  SKU-10094         95           270    Delivered
265  PO/2026/032155  SKU-10594      

In [9]:
df_validated = df_numeric.copy()

# 1. Flag negative values
negative_cols = [
    "order_qty",
    "unit_cost_inr",
    "on_hand_stock",
    "safety_stock",
    "avg_daily_demand",
    "freight_cost_inr"
]

for col in negative_cols:
    df_validated[f"flag_negative_{col}"] = df_validated[col] < 0

# 2. Flag invalid GST rates
valid_gst_rates = [0, 5, 12, 18, 28]

df_validated["flag_invalid_gst"] = (
    df_validated["gst_rate_pct"].notna()
    & ~df_validated["gst_rate_pct"].isin(valid_gst_rates)
)

# 3. Flag received quantity greater than ordered quantity
df_validated["flag_received_gt_ordered"] = (
    df_validated["received_qty"].notna()
    & df_validated["order_qty"].notna()
    & (df_validated["received_qty"] > df_validated["order_qty"])
)

# 4. Flag zero quantities and costs
for col in ["order_qty", "unit_cost_inr", "avg_daily_demand"]:
    df_validated[f"flag_zero_{col}"] = df_validated[col] == 0

# 5. Summary of flagged records
flag_cols = [col for col in df_validated.columns if col.startswith("flag_")]

print("Flag counts:")
print(df_validated[flag_cols].sum().sort_values(ascending=False))

print("\nDataset shape:", df_validated.shape)

Flag counts:
flag_received_gt_ordered          2838
flag_invalid_gst                  1647
flag_negative_order_qty            622
flag_negative_on_hand_stock        523
flag_negative_safety_stock         507
flag_negative_avg_daily_demand     357
flag_zero_avg_daily_demand         323
flag_zero_order_qty                292
flag_negative_freight_cost_inr     215
flag_zero_unit_cost_inr            208
flag_negative_unit_cost_inr        177
dtype: Int64

Dataset shape: (102300, 37)


In [10]:
date_cols = ["order_date", "promised_date", "received_date"]

for col in date_cols:
    print(f"\n{'=' * 20} {col} {'=' * 20}")
    print("Data type:", df_validated[col].dtype)
    print("Missing values:", df_validated[col].isna().sum())

    values = (
        df_validated[col]
        .dropna()
        .astype("string")
        .str.strip()
    )

    print("Sample values:")
    print(values.sample(min(15, len(values)), random_state=42).to_list())

    print("\nMost common raw formats/values:")
    print(values.value_counts().head(15))


==================== order_date ====================
Data type: str
Missing values: 127
Sample values:
['2025-02-11', '26-08-2026', '2024-03-30', '2025-05-09', '2025-08-23', '10/18/2025', '2023-10-24', '2024-11-21', '2026-04-22', '2025-08-05', '2024-06-25', '2025-08-28', '2025-08-02', '05/08/2025', '2026-08-06']

Most common raw formats/values:
order_date
??            135
TBD           113
0000-00-00    111
2024-03-17     76
2025-01-30     73
2024-07-02     72
2023-11-02     72
2026-08-05     71
2024-06-14     71
2025-01-06     70
2024-04-16     70
2025-08-19     70
2024-01-28     68
2023-10-15     68
2026-01-04     68
Name: count, dtype: int64[pyarrow]

==================== promised_date ====================
Data type: str
Missing values: 0
Sample values:
['2026-03-03', '02-07-2024', '2023-09-16', '10/10/2025', '16-12-2024', '2024-10-24', '2023-09-09', '2024-03-13', '2026-01-24', '2023-10-17', '02.06.2024', '2024-03-22', '13-Sep-26', '2023-06-08', '2026-09-04']

Most common raw form

In [11]:
import pandas as pd

df_dates = df_validated.copy()

date_cols = ["order_date", "promised_date", "received_date"]

def parse_mixed_date(series):
    s = (
        series.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "??": pd.NA,
            "TBD": pd.NA,
            "0000-00-00": pd.NA,
            "null": pd.NA,
            "NA": pd.NA,
            "-": pd.NA
        })
    )

    # Parse each value individually so formats can vary.
    def parse_one(value):
        if pd.isna(value):
            return pd.NaT

        value = str(value).strip()

        formats = [
            "%Y-%m-%d",   # 2025-02-11
            "%d-%m-%Y",   # 26-08-2026
            "%d/%m/%Y",   # 05/08/2025 (day-first)
            "%m/%d/%Y",   # US-style, e.g. 10/18/2025
            "%d.%m.%Y",   # 02.06.2024
            "%d-%b-%y",   # 13-Sep-26
            "%d-%b-%Y",   # 13-Sep-2026
        ]

        for fmt in formats:
            try:
                return pd.to_datetime(value, format=fmt, errors="raise")
            except (ValueError, TypeError):
                continue

        return pd.NaT

    return pd.to_datetime(s.map(parse_one))


for col in date_cols:
    df_dates[f"{col}_parsed"] = parse_mixed_date(df_dates[col])

    print(f"\n{col}")
    print("Parsed missing/invalid:", df_dates[f"{col}_parsed"].isna().sum())

    # Show non-placeholder values that could not be parsed
    original = df_dates[col].astype("string").str.strip()
    placeholders = ["", "??", "TBD", "0000-00-00", "null", "NA", "-"]

    failed = (
        original.notna()
        & ~original.isin(placeholders)
        & df_dates[f"{col}_parsed"].isna()
    )

    print("Unparsed examples:")
    print(original[failed].value_counts().head(10))

print("\nDataset shape:", df_dates.shape)


order_date
Parsed missing/invalid: 486
Unparsed examples:
Series([], Name: count, dtype: int64[pyarrow])

promised_date
Parsed missing/invalid: 1001
Unparsed examples:
promised_date
31/02/2025    1001
Name: count, dtype: int64[pyarrow]

received_date
Parsed missing/invalid: 13541
Unparsed examples:
Series([], Name: count, dtype: int64[pyarrow])

Dataset shape: (102300, 40)


In [12]:
# Order date after promised date
mask = (
    df_dates["order_date_parsed"].notna()
    & df_dates["promised_date_parsed"].notna()
    & (df_dates["promised_date_parsed"] < df_dates["order_date_parsed"])
)
print("Promised date before order date:", mask.sum())

# Received date before order date
mask = (
    df_dates["received_date_parsed"].notna()
    & df_dates["order_date_parsed"].notna()
    & (df_dates["received_date_parsed"] < df_dates["order_date_parsed"])
)
print("Received date before order date:", mask.sum())

# Received date on cancelled orders
mask = (
    df_dates["status"].eq("Cancelled")
    & df_dates["received_date_parsed"].notna()
)
print("Cancelled orders with received date:", mask.sum())

# Suspicious default received date
mask = df_dates["received_date_parsed"].eq(pd.Timestamp("2020-01-01"))
print("Received date = 2020-01-01:", mask.sum())

Promised date before order date: 2741
Received date before order date: 3764
Cancelled orders with received date: 1554
Received date = 2020-01-01: 1518


In [13]:
df_dates["flag_promised_before_order"] = (
    df_dates["order_date_parsed"].notna()
    & df_dates["promised_date_parsed"].notna()
    & (df_dates["promised_date_parsed"] < df_dates["order_date_parsed"])
)

df_dates["flag_received_before_order"] = (
    df_dates["order_date_parsed"].notna()
    & df_dates["received_date_parsed"].notna()
    & (df_dates["received_date_parsed"] < df_dates["order_date_parsed"])
)

df_dates["flag_cancelled_with_received_date"] = (
    df_dates["status"].eq("Cancelled")
    & df_dates["received_date_parsed"].notna()
)

df_dates["flag_default_received_date"] = (
    df_dates["received_date_parsed"].eq(pd.Timestamp("2020-01-01"))
)

# Inspect suspicious records
flag_cols = [
    "flag_promised_before_order",
    "flag_received_before_order",
    "flag_cancelled_with_received_date",
    "flag_default_received_date"
]

print("Flag counts:")
print(df_dates[flag_cols].sum())

print("\nExamples of date conflicts:")
print(
    df_dates.loc[
        df_dates[flag_cols].any(axis=1),
        [
            "order_id", "status",
            "order_date", "promised_date", "received_date",
            "order_date_parsed", "promised_date_parsed",
            "received_date_parsed", "order_qty", "received_qty"
        ]
    ].head(20).to_string(index=False)
)

print("\nDataset shape:", df_dates.shape)

Flag counts:
flag_promised_before_order           2741
flag_received_before_order           3764
flag_cancelled_with_received_date    1554
flag_default_received_date           1518
dtype: int64

Examples of date conflicts:
      order_id      status order_date promised_date received_date order_date_parsed promised_date_parsed received_date_parsed  order_qty  received_qty
PO/2024/034209   Delivered 2024-10-27    2024-11-19    2020-01-01        2024-10-27           2024-11-19           2020-01-01        369           295
PO/2026/054586   Delivered 2026-01-01    01/18/2026    2020-01-01        2026-01-01           2026-01-18           2020-01-01        198           178
PO/2024/093273   Cancelled 06/03/2024    2024-03-25    2024-03-15        2024-03-06           2024-03-25           2024-03-15        699           699
PO/2026/035745   Delivered 2026-07-21    2026-08-01    08/04/2026        2026-07-21           2026-08-01           2026-04-08        173           173
PO/2025/028311 Backord

In [14]:
import pandas as pd

df_dates_v2 = df_validated.copy()

def parse_date_consistent(series):
    s = (
        series.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "??": pd.NA,
            "TBD": pd.NA,
            "0000-00-00": pd.NA,
            "null": pd.NA,
            "NA": pd.NA,
            "-": pd.NA
        })
    )

    def parse_one(value):
        if pd.isna(value):
            return pd.NaT

        value = str(value).strip()

        # ISO format
        try:
            return pd.to_datetime(value, format="%Y-%m-%d", errors="raise")
        except ValueError:
            pass

        # Day-first formats
        for fmt in [
            "%d-%m-%Y",
            "%d/%m/%Y",
            "%d.%m.%Y",
            "%d-%b-%y",
            "%d-%b-%Y"
        ]:
            try:
                return pd.to_datetime(value, format=fmt, errors="raise")
            except ValueError:
                pass

        # US-style slash format, only needed if day-first is impossible
        try:
            return pd.to_datetime(value, format="%m/%d/%Y", errors="raise")
        except ValueError:
            return pd.NaT

    return pd.to_datetime(s.map(parse_one))


for col in ["order_date", "promised_date", "received_date"]:
    df_dates_v2[f"{col}_parsed"] = parse_date_consistent(df_dates_v2[col])

    print(
        col,
        "missing/invalid:",
        df_dates_v2[f"{col}_parsed"].isna().sum()
    )

print("Shape:", df_dates_v2.shape)

order_date missing/invalid: 486
promised_date missing/invalid: 1001
received_date missing/invalid: 13541
Shape: (102300, 40)


In [15]:
# Promised date before order date
promised_before = (
    df_dates_v2["order_date_parsed"].notna()
    & df_dates_v2["promised_date_parsed"].notna()
    & (
        df_dates_v2["promised_date_parsed"]
        < df_dates_v2["order_date_parsed"]
    )
)

# Received date before order date
received_before = (
    df_dates_v2["order_date_parsed"].notna()
    & df_dates_v2["received_date_parsed"].notna()
    & (
        df_dates_v2["received_date_parsed"]
        < df_dates_v2["order_date_parsed"]
    )
)

# Cancelled orders with received dates
cancelled_received = (
    df_dates_v2["status"].eq("Cancelled")
    & df_dates_v2["received_date_parsed"].notna()
)

# Suspicious default date
default_received = df_dates_v2["received_date_parsed"].eq(
    pd.Timestamp("2020-01-01")
)

print("Promised date before order date:", promised_before.sum())
print("Received date before order date:", received_before.sum())
print("Cancelled orders with received date:", cancelled_received.sum())
print("Received date = 2020-01-01:", default_received.sum())

Promised date before order date: 2741
Received date before order date: 3764
Cancelled orders with received date: 1554
Received date = 2020-01-01: 1518


In [16]:
# 1. Exact duplicate rows
exact_dup_mask = df_dates_v2.duplicated(keep=False)
print("Rows involved in exact duplicates:", exact_dup_mask.sum())
print(
    "Extra duplicate rows to remove (keep first):",
    df_dates_v2.duplicated().sum()
)

# 2. Repeated order IDs
order_id = df_dates_v2["order_id"].astype("string").str.strip()

repeated_id_mask = (
    order_id.notna()
    & order_id.ne("")
    & order_id.duplicated(keep=False)
)

print("\nRows with repeated order IDs:", repeated_id_mask.sum())
print("Number of repeated order IDs:", order_id[repeated_id_mask].nunique())

# 3. Inspect repeated order IDs
repeated_orders = df_dates_v2.loc[
    repeated_id_mask,
    [
        "order_id", "sku", "product_name", "supplier",
        "order_date", "order_qty", "received_qty", "status"
    ]
].sort_values(["order_id", "sku"])

print("\nExamples of repeated order IDs:")
print(repeated_orders.head(30).to_string(index=False))

# 4. Inspect exact duplicates
print("\nExamples of exact duplicate rows:")
print(
    df_dates_v2.loc[exact_dup_mask]
    .head(10)
    .to_string(index=False)
)

Rows involved in exact duplicates: 710
Extra duplicate rows to remove (keep first): 355

Rows with repeated order IDs: 3002
Number of repeated order IDs: 1501

Examples of repeated order IDs:
      order_id       sku             product_name                supplier order_date  order_qty  received_qty             status
PO/2023/000083 SKU-10288  Tea Powder 500g - Var13    Jai Hind Enterprises 2023-12-09        221           198          Delivered
PO/2023/000083 SKU-10288  Tea Powder 500g - Var13    Jai Hind Enterprises 2023-12-09        221           198          Delivered
PO/2023/000755 SKU-10502      Sanitizer 5L - Var1    Tirupati Steel Works 2023-05-26         60           144          Delivered
PO/2023/000755 SKU-10502      Sanitizer 5L - Var1    Tirupati Steel Works 2023-05-26         60           144          Delivered
PO/2023/001286 SKU-10519        ORS Carton - Var4          Mahalaxmi FMCG 2023-09-11        330           313        Backordered
PO/2023/001286 SKU-10519        OR

In [17]:
# Keep one copy of each fully identical row
df_deduplicated = df_dates_v2.drop_duplicates(keep="first").copy()

print("Original rows:", len(df_dates_v2))
print("Rows after removing exact duplicates:", len(df_deduplicated))
print("Rows removed:", len(df_dates_v2) - len(df_deduplicated))
print("Remaining exact duplicates:", df_deduplicated.duplicated().sum())
print("Dataset shape:", df_deduplicated.shape)

Original rows: 102300
Rows after removing exact duplicates: 101945
Rows removed: 355
Remaining exact duplicates: 0
Dataset shape: (101945, 40)


In [18]:
order_id = (
    df_deduplicated["order_id"]
    .astype("string")
    .str.strip()
)

valid_id = order_id.notna() & order_id.ne("")
repeated_mask = valid_id & order_id.duplicated(keep=False)

print("Rows with repeated order IDs:", repeated_mask.sum())
print("Number of repeated order IDs:", order_id[repeated_mask].nunique())

print("\nExamples:")
print(
    df_deduplicated.loc[
        repeated_mask,
        [
            "order_id", "sku", "product_name", "supplier",
            "order_date", "order_qty", "received_qty", "status"
        ]
    ]
    .sort_values(["order_id", "sku"])
    .head(30)
    .to_string(index=False)
)

Rows with repeated order IDs: 2292
Number of repeated order IDs: 1146

Examples:
      order_id       sku               product_name                  supplier order_date  order_qty  received_qty             status
PO/2023/001286 SKU-10519          ORS Carton - Var4            Mahalaxmi FMCG 2023-09-11        330           313        Backordered
PO/2023/001286 SKU-10519          ORS Carton - Var4            Mahalaxmi FMCG 2023-09-11        330           313        Backordered
PO/2023/001348 SKU-10239     Tea Powder 500g - Var6         Shakti Components 12.05.2023        252           226          Delivered
PO/2023/001348 SKU-10239     Tea Powder 500g - Var6         Shakti Components 12.05.2023        252           226          Delivered
PO/2023/001954 SKU-10128       Winter Jacket - Var5   Rajdhani Wholesale Mart 12/05/2023        980           960 Partially Received
PO/2023/001954 SKU-10128       Winter Jacket - Var5   Rajdhani Wholesale Mart 12/05/2023        980           960 Partial

In [19]:
# 1. Check exact duplicates across all columns
print("Exact duplicate rows remaining:",
      df_deduplicated.duplicated().sum())

# 2. Inspect one repeated order ID across every column
sample_id = "PO/2023/001286"

sample = df_deduplicated[
    df_deduplicated["order_id"].astype("string").str.strip() == sample_id
]

print("\nRows for sample order ID:", len(sample))
print(sample.T.to_string())

# 3. Check whether the two rows differ in any column
if len(sample) >= 2:
    row1 = sample.iloc[0]
    row2 = sample.iloc[1]

    different_cols = [
        col for col in sample.columns
        if not (
            (pd.isna(row1[col]) and pd.isna(row2[col]))
            or row1[col] == row2[col]
        )
    ]

    print("\nColumns that differ:", different_cols)

Exact duplicate rows remaining: 0

Rows for sample order ID: 2
                                                        41243                          51648
order_id                                       PO/2023/001286                 PO/2023/001286
sku                                                 SKU-10519                      SKU-10519
product_name                                ORS Carton - Var4              ORS Carton - Var4
category                                      Pharma & Health                Pharma & Health
hsn_code                                               300463                         300463
supplier                                       Mahalaxmi FMCG                 Mahalaxmi FMCG
supplier_gstin                                33QXABB7325S2ZN                33QXABB7325S2ZN
supplier_city                                      Coimbatore                     Coimbatore
warehouse                                    Lucknow (Amausi)               Lucknow (Amausi)
warehou

In [20]:
# Find rows with repeated, non-missing order IDs
order_id = df_deduplicated["order_id"].astype("string").str.strip()

valid_id = order_id.notna() & order_id.ne("")
repeated_mask = valid_id & order_id.duplicated(keep=False)

repeated_orders = df_deduplicated.loc[repeated_mask].copy()

# Count repeated order IDs
print("Rows with repeated order IDs:", len(repeated_orders))
print(
    "Unique repeated order IDs:",
    order_id[repeated_mask].nunique()
)

# Count distinct records for each repeated order ID
distinct_records = (
    repeated_orders
    .groupby("order_id", dropna=False)
    .apply(lambda group: len(group.drop_duplicates()), include_groups=False)
)

print("\nRepeated order IDs by number of distinct records:")
print(distinct_records.value_counts().sort_index())

Rows with repeated order IDs: 2292
Unique repeated order IDs: 1146

Repeated order IDs by number of distinct records:
2    1146
Name: count, dtype: int64


In [21]:
order_id = df_deduplicated["order_id"].astype("string").str.strip()
valid_mask = order_id.notna() & order_id.ne("")

repeated_mask = valid_mask & order_id.duplicated(keep=False)
repeated_orders = df_deduplicated.loc[repeated_mask].copy()

# Exclude technical validation flags and parsed helper columns
business_cols = [
    col for col in repeated_orders.columns
    if not col.startswith("flag_") and not col.endswith("_parsed")
]

difference_counts = {}

for order, group in repeated_orders.groupby("order_id", sort=False):
    if len(group) != 2:
        continue

    row1, row2 = group.iloc[0], group.iloc[1]

    for col in business_cols:
        a, b = row1[col], row2[col]

        both_missing = pd.isna(a) and pd.isna(b)
        equal = both_missing or (not pd.isna(a) and not pd.isna(b) and a == b)

        if not equal:
            difference_counts[col] = difference_counts.get(col, 0) + 1

print("Number of repeated order IDs:", repeated_orders["order_id"].nunique())
print("\nNumber of pairs differing by column:")
print(
    pd.Series(difference_counts, name="pairs")
    .sort_values(ascending=False)
)

Number of repeated order IDs: 1146

Number of pairs differing by column:
notes               1146
sku                    1
product_name           1
category               1
hsn_code               1
supplier               1
supplier_gstin         1
supplier_city          1
warehouse              1
warehouse_state        1
order_date             1
promised_date          1
received_date          1
order_qty              1
received_qty           1
unit_cost_inr          1
on_hand_stock          1
reorder_point          1
safety_stock           1
avg_daily_demand       1
freight_cost_inr       1
transporter            1
payment_terms          1
status                 1
Name: pairs, dtype: int64


In [22]:
import pandas as pd

order_id = df_deduplicated["order_id"].astype("string").str.strip()

repeated_mask = (
    order_id.notna()
    & order_id.ne("")
    & order_id.duplicated(keep=False)
)

repeated_orders = df_deduplicated.loc[repeated_mask].copy()

business_cols = [
    col for col in repeated_orders.columns
    if col != "notes"
    and not col.startswith("flag_")
    and not col.endswith("_parsed")
]

exceptional_ids = []

for oid, group in repeated_orders.groupby(
    repeated_orders["order_id"].astype("string").str.strip(),
    sort=False
):
    if len(group) != 2:
        continue

    row1, row2 = group.iloc[0], group.iloc[1]

    different_cols = []

    for col in business_cols:
        a, b = row1[col], row2[col]

        both_missing = pd.isna(a) and pd.isna(b)
        equal = both_missing or (
            not pd.isna(a) and not pd.isna(b) and a == b
        )

        if not equal:
            different_cols.append(col)

    if different_cols:
        exceptional_ids.append((oid, different_cols))

print("Exceptional order IDs:", len(exceptional_ids))

for oid, cols in exceptional_ids:
    print("\nOrder ID:", oid)
    print("Different business columns:", cols)

    display(
        repeated_orders[
            repeated_orders["order_id"].astype("string").str.strip() == oid
        ].T
    )

Exceptional order IDs: 1

Order ID: PO/2024/952594
Different business columns: ['sku', 'product_name', 'category', 'hsn_code', 'supplier', 'supplier_gstin', 'supplier_city', 'warehouse', 'warehouse_state', 'order_date', 'promised_date', 'received_date', 'order_qty', 'received_qty', 'unit_cost_inr', 'on_hand_stock', 'reorder_point', 'safety_stock', 'avg_daily_demand', 'freight_cost_inr', 'transporter', 'payment_terms', 'status']


,5030,60788
order_id,PO/2024/952594,PO/2024/952594
sku,SKU-10367,SKU-10059
product_name,Mixer Grinder - Var12,Power Bank - Var9
category,Home & Kitchen,Electronics
hsn_code,732322,851743
supplier,balaji packaging,Shakti Components
supplier_gstin,24RJTQA9932K3ZA,32GATXA5367T1ZB
supplier_city,Ahmedabad,Kochi
warehouse,Lucknow (Amausi),Mumbai (Bhiwandi)
warehouse_state,Uttar Pradesh,Maharashtra


In [23]:
df_order_clean = df_deduplicated.copy()

# Identify repeated order IDs
ids = df_order_clean["order_id"].astype("string").str.strip()
valid = ids.notna() & ids.ne("")
repeated_ids = ids[valid & ids.duplicated(keep=False)].unique()

# Columns that must match before consolidation
compare_cols = [
    col for col in df_order_clean.columns
    if col != "notes"
]

mergeable_ids = []
review_ids = []

for oid in repeated_ids:
    group = df_order_clean.loc[ids == oid]

    if len(group) != 2:
        review_ids.append(oid)
        continue

    # Check all columns except notes
    if group[compare_cols].nunique(dropna=False).eq(1).all():
        mergeable_ids.append(oid)
    else:
        review_ids.append(oid)

print("Pairs safe to consider for note consolidation:", len(mergeable_ids))
print("Order IDs requiring review:", len(review_ids))
print("Example requiring review:", review_ids[:5])

Pairs safe to consider for note consolidation: 1145
Order IDs requiring review: 1
Example requiring review: ['PO/2024/952594']


In [25]:
df_order_clean = df_deduplicated.copy()

# Normalize order IDs for matching
ids = df_order_clean["order_id"].astype("string").str.strip()

# IDs verified as identical across every column except notes
mergeable_set = set(mergeable_ids)

# Keep one row per verified pair and combine its notes
rows_to_keep = []
merged_notes = {}

for oid in mergeable_ids:
    group = df_order_clean.loc[ids == oid]

    notes = (
        group["notes"]
        .dropna()
        .astype("string")
        .str.strip()
    )

    notes = notes[notes.ne("")].drop_duplicates()

    merged_notes[oid] = " | ".join(notes.tolist()) if len(notes) else pd.NA

    # Keep the first row; its non-note columns match the other row
    rows_to_keep.append(group.index[0])

# Keep all non-mergeable rows, including the exceptional pair
mergeable_mask = ids.isin(mergeable_set)
keep_mask = ~mergeable_mask | df_order_clean.index.isin(rows_to_keep)

df_order_clean = df_order_clean.loc[keep_mask].copy()

# Restore combined notes on retained rows
clean_ids = df_order_clean["order_id"].astype("string").str.strip()

for oid, note in merged_notes.items():
    df_order_clean.loc[clean_ids == oid, "notes"] = note

print("Rows before consolidation:", len(df_deduplicated))
print("Rows after consolidation:", len(df_order_clean))
print("Rows removed:", len(df_deduplicated) - len(df_order_clean))
print(
    "Repeated order IDs remaining:",
    df_order_clean.loc[
        df_order_clean["order_id"].astype("string").str.strip()
        .duplicated(keep=False),
        "order_id"
    ].nunique()
)

Rows before consolidation: 101945
Rows after consolidation: 100800
Rows removed: 1145
Repeated order IDs remaining: 1


In [26]:
print("Dataset shape:", df_order_clean.shape)
print("Exact duplicate rows:", df_order_clean.duplicated().sum())

ids = df_order_clean["order_id"].astype("string").str.strip()
repeated = ids.notna() & ids.ne("") & ids.duplicated(keep=False)

print("Rows with repeated order IDs:", repeated.sum())
print("Repeated order IDs:", ids[repeated].nunique())
print("\nRemaining repeated order ID:")
print(df_order_clean.loc[repeated, ["order_id", "sku", "product_name", "status", "notes"]])

Dataset shape: (100800, 40)
Exact duplicate rows: 0
Rows with repeated order IDs: 2
Repeated order IDs: 1

Remaining repeated order ID:
             order_id        sku           product_name              status  \
5030   PO/2024/952594  SKU-10367  Mixer Grinder - Var12           Delivered   
60788  PO/2024/952594  SKU-10059      Power Bank - Var9  Partially Received   

                            notes  
5030                          NaN  
60788  monsoon delay - NH blocked  


In [27]:
# Inspect the remaining repeated order ID
target_id = "PO/2024/952594"

order_check = df_order_clean[
    df_order_clean["order_id"].astype("string").str.strip() == target_id
].copy()

cols_to_check = [
    "order_id",
    "sku",
    "product_name",
    "supplier",
    "supplier_gstin",
    "warehouse",
    "order_date",
    "promised_date",
    "received_date",
    "order_qty",
    "received_qty",
    "status",
    "notes"
]

print("Number of matching rows:", len(order_check))
display(order_check[cols_to_check])

Number of matching rows: 2


,order_id,sku,product_name,supplier,supplier_gstin,warehouse,order_date,promised_date,received_date,order_qty,received_qty,status,notes
5030,PO/2024/952594,SKU-10367,Mixer Grinder - Var12,balaji packaging,24RJTQA9932K3ZA,Lucknow (Amausi),12/07/2025,13-12-2025,2025-12-11,150,147,Delivered,NaN
60788,PO/2024/952594,SKU-10059,Power Bank - Var9,Shakti Components,32GATXA5367T1ZB,Mumbai (Bhiwandi),2025-01-25,2025-01-29,2025-01-27,99,89,Partially Received,monsoon delay - NH blocked


In [28]:
for sku in order_check["sku"].dropna().unique():
    print(f"\nRecords for SKU: {sku}")

    display(
        df_order_clean.loc[
            df_order_clean["sku"].eq(sku),
            ["order_id", "sku", "supplier", "order_date",
             "order_qty", "received_qty", "status"]
        ].head(10)
    )


Records for SKU: SKU-10367


,order_id,sku,supplier,order_date,order_qty,received_qty,status
858,PO/2023/059065,SKU-10367,Reliable Industrial Corp,28.09.2023,116,116,Delivered
1943,PO/2023/059571,SKU-10367,MARUTI LOGISTICS PVT LTD,23.12.2023,478,468,Delivered
2432,PO/2025/020205,SKU-10367,Patel & Sons,24-Jun-25,44,105,Cancelled
3863,PO/2024/096739,SKU-10367,Narmada Polymers,2024-02-20,371,363,Delivered
4809,PO/2024/054465,SKU-10367,Kisan Agri Hub,08/03/2024,2002,2002,Delivered
5030,PO/2024/952594,SKU-10367,balaji packaging,12/07/2025,150,147,Delivered
6391,PO/2024/021258,SKU-10367,Jai Hind Enterprises,2024-05-07,976,976,Delivered
10159,PO/2025/023216,SKU-10367,Surya Electricals,11-11-2025,241,236,Delivered
10236,PO/2026/006831,SKU-10367,Sri Lakshmi Distributors,2026-04-02,368,<NA>,Delivered
11325,PO/2026/047523,SKU-10367,Rajdhani Wholesale Mart,2026-09-09,93,74,Delivered



Records for SKU: SKU-10059


,order_id,sku,supplier,order_date,order_qty,received_qty,status
24,PO/2024/085202,SKU-10059,Ambika Pharma Distributors,05/22/2024,147,132,Delivered
434,PO/2025/016202,SKU-10059,Shakti Components,2025-01-25,99,89,Partially Received
935,PO/2023/007739,SKU-10059,Anand Cotton Mills,2023-07-31,83,83,Delivered
2390,PO/2024/086925,SKU-10059,Surya Electricals,11/02/2024,113,113,Delivered
4486,PO/2023/082623,SKU-10059,Ambika Pharma Distributors,22.04.2023,219,219,Delivered
4892,PO/2026/002582,SKU-10059,Patel & Sons,2026-05-07,105,94,Delivered
5339,PO/2025/000995,SKU-10059,Malabar Spices & Foods,05-Nov-25,81,64,Delivered
5487,PO/2026/017552,SKU-10059,Dhanlaxmi Home Products,27-Apr-26,224,219,Partially Received
5514,PO/2024/086992,SKU-10059,mahalaxmi fmcg,2024-06-30,112,89,Delivered
6449,PO/2025/006654,SKU-10059,Kaveri Textiles,03-Aug-25,480,384,Delivered


In [29]:
date_cols = [
    "order_date_parsed",
    "promised_date_parsed",
    "received_date_parsed"
]

for col in date_cols:
    print(f"\n{col}")
    print("Missing or unparsed:", df_order_clean[col].isna().sum())
    print("Earliest date:", df_order_clean[col].min())
    print("Latest date:", df_order_clean[col].max())


order_date_parsed
Missing or unparsed: 479
Earliest date: 2023-01-04 00:00:00
Latest date: 2026-12-09 00:00:00

promised_date_parsed
Missing or unparsed: 983
Earliest date: 2023-01-05 00:00:00
Latest date: 2026-12-09 00:00:00

received_date_parsed
Missing or unparsed: 13362
Earliest date: 2020-01-01 00:00:00
Latest date: 2026-12-10 00:00:00


In [30]:
df_date_clean = df_order_clean.copy()

# Order date should not be after promised date
df_date_clean["flag_promised_before_order"] = (
    df_date_clean["order_date_parsed"].notna()
    & df_date_clean["promised_date_parsed"].notna()
    & (
        df_date_clean["promised_date_parsed"]
        < df_date_clean["order_date_parsed"]
    )
)

# Receipt should not precede order
df_date_clean["flag_received_before_order"] = (
    df_date_clean["received_date_parsed"].notna()
    & df_date_clean["order_date_parsed"].notna()
    & (
        df_date_clean["received_date_parsed"]
        < df_date_clean["order_date_parsed"]
    )
)

# Cancelled orders should be checked if a receipt date exists
df_date_clean["flag_cancelled_with_receipt_date"] = (
    df_date_clean["status"].astype("string").str.casefold().eq("cancelled")
    & df_date_clean["received_date_parsed"].notna()
)

# Investigate suspicious default dates
df_date_clean["flag_suspected_default_received_date"] = (
    df_date_clean["received_date_parsed"]
    .eq(pd.Timestamp("2020-01-01"))
    .fillna(False)
)

flag_cols = [
    "flag_promised_before_order",
    "flag_received_before_order",
    "flag_cancelled_with_receipt_date",
    "flag_suspected_default_received_date"
]

print("Date validation flags:")
print(df_date_clean[flag_cols].sum().sort_values(ascending=False))

Date validation flags:
flag_received_before_order              3720.0
flag_promised_before_order              2711.0
flag_cancelled_with_receipt_date        1539.0
flag_suspected_default_received_date    1500.0
dtype: double[pyarrow]


In [32]:
for raw_col, parsed_col in [
    ("order_date", "order_date_parsed"),
    ("promised_date", "promised_date_parsed"),
    ("received_date", "received_date_parsed")
]:
    raw = df_date_clean[raw_col].astype("string").str.strip()

    invalid = raw.notna() & raw.ne("") & df_date_clean[parsed_col].isna()

    print(f"\n{raw_col}")
    print("Non-empty values that failed parsing:", invalid.sum())
    print("Examples:")
    print(raw[invalid].value_counts().head(10))


order_date
Non-empty values that failed parsing: 353
Examples:
order_date
??            133
0000-00-00    110
TBD           110
Name: count, dtype: int64[pyarrow]

promised_date
Non-empty values that failed parsing: 983
Examples:
promised_date
31/02/2025    983
Name: count, dtype: int64[pyarrow]

received_date
Non-empty values that failed parsing: 0
Examples:
Series([], Name: count, dtype: int64[pyarrow])


In [33]:
df_dates_clean = df_date_clean.copy()

# Mark invalid parsed dates as missing
for col in [
    "order_date_parsed",
    "promised_date_parsed",
    "received_date_parsed"
]:
    df_dates_clean[col] = pd.to_datetime(
        df_dates_clean[col],
        errors="coerce"
    )

# Count missing parsed dates
for col in [
    "order_date_parsed",
    "promised_date_parsed",
    "received_date_parsed"
]:
    print(f"{col}: {df_dates_clean[col].isna().sum()} missing dates")

print("\nDataset shape:", df_dates_clean.shape)

order_date_parsed: 479 missing dates
promised_date_parsed: 983 missing dates
received_date_parsed: 13362 missing dates

Dataset shape: (100800, 44)


In [34]:
print(
    "Promised date before order date:",
    df_dates_clean["flag_promised_before_order"].sum()
)

print(
    "Received date before order date:",
    df_dates_clean["flag_received_before_order"].sum()
)

print(
    "Cancelled orders with receipt dates:",
    df_dates_clean["flag_cancelled_with_receipt_date"].sum()
)

print(
    "Suspected default receipt dates:",
    df_dates_clean["flag_suspected_default_received_date"].sum()
)

Promised date before order date: 2711
Received date before order date: 3720
Cancelled orders with receipt dates: 1539
Suspected default receipt dates: 1500


In [35]:
# Columns useful for investigating date conflicts
inspect_cols = [
    "order_id",
    "supplier",
    "sku",
    "order_date",
    "promised_date",
    "received_date",
    "order_date_parsed",
    "promised_date_parsed",
    "received_date_parsed",
    "order_qty",
    "received_qty",
    "status",
    "notes"
]

# 1. Promised date before order date
print("=== Promised date before order date ===")
display(
    df_dates_clean.loc[
        df_dates_clean["flag_promised_before_order"],
        inspect_cols
    ].head(10)
)

# 2. Received date before order date
print("=== Received date before order date ===")
display(
    df_dates_clean.loc[
        df_dates_clean["flag_received_before_order"],
        inspect_cols
    ].head(10)
)

# 3. Cancelled orders with receipt dates
print("=== Cancelled orders with receipt dates ===")
display(
    df_dates_clean.loc[
        df_dates_clean["flag_cancelled_with_receipt_date"],
        inspect_cols
    ].head(10)
)

# 4. Suspected default receipt dates
print("=== Suspected default receipt dates ===")
display(
    df_dates_clean.loc[
        df_dates_clean["flag_suspected_default_received_date"],
        inspect_cols
    ].head(10)
)

=== Promised date before order date ===


,order_id,supplier,sku,order_date,promised_date,received_date,order_date_parsed,promised_date_parsed,received_date_parsed,order_qty,received_qty,status,notes
22,PO/2025/028311,Rajdhani Wholesale Mart,SKU-10545,2025-11-12,12/06/2025,NaN,2025-11-12,2025-06-12,NaT,72,72,Backordered,TBD
40,PO/2024/027704,Anand Cotton Mills,SKU-10547,10/03/2024,04/03/2024,NaN,2024-03-10,2024-03-04,NaT,184,184,Backordered,recheck count
44,PO/2026/081063,Deccan Traders,SKU-10096,02/05/2026,20-Feb-26,NaN,2026-05-02,2026-02-20,NaT,5,12,Cancelled,LR copy missing
46,PO/2024/085116,Om Sai Trading Co,SKU-10202,07/08/2024,2024-07-26,2024-08-03,2024-08-07,2024-07-26,2024-08-03,126,126,Delivered,urgent
51,PO/2025/087197,Ganesh Agro Supplies,SKU-10506,03/07/2025,2025-03-18,2025-03-12,2025-07-03,2025-03-18,2025-03-12,248,198,Delivered,NaN
97,PO/2025/094683,Mahalaxmi FMCG,SKU-10290,16/07/2025,08/06/2025,30-Jul-25,2025-07-16,2025-06-08,2025-07-30,164,147,Delivered,NaN
125,PO/2025/046663,Deccan Traders,SKU-10693,04/11/2025,2025-04-30,2025-04-30,2025-11-04,2025-04-30,2025-04-30,39,35,Delivered,Diwali rush
265,PO/2026/032155,Reliable Industrial Corp,SKU-10594,2026-07-29,08/02/2026,NaN,2026-07-29,2026-02-08,NaT,85,240,In Transit,NaN
313,PO/2024/022539,Bharat Traders,SKU-10242,2024-08-01,08/05/2024,2024-08-11,2024-08-01,2024-05-08,2024-08-11,135,135,Delivered,monsoon delay - NH blocked
353,PO/2023/078883,Surya Electricals,SKU-10464,2023-11-23,12/07/2023,NaN,2023-11-23,2023-07-12,NaT,74,66,Backordered,NaN


=== Received date before order date ===


,order_id,supplier,sku,order_date,promised_date,received_date,order_date_parsed,promised_date_parsed,received_date_parsed,order_qty,received_qty,status,notes
1,PO/2024/034209,Himalaya Wholesale,SKU-10170,2024-10-27,2024-11-19,2020-01-01,2024-10-27,2024-11-19,2020-01-01,369,295,Delivered,NaN
14,PO/2026/054586,Surya Electricals,SKU-10270,2026-01-01,01/18/2026,2020-01-01,2026-01-01,2026-01-18,2020-01-01,198,178,Delivered,NaN
19,PO/2026/035745,Dhanlaxmi Home Products,SKU-10311,2026-07-21,2026-08-01,08/04/2026,2026-07-21,2026-08-01,2026-04-08,173,173,Delivered,NaN
46,PO/2024/085116,Om Sai Trading Co,SKU-10202,07/08/2024,2024-07-26,2024-08-03,2024-08-07,2024-07-26,2024-08-03,126,126,Delivered,urgent
51,PO/2025/087197,Ganesh Agro Supplies,SKU-10506,03/07/2025,2025-03-18,2025-03-12,2025-07-03,2025-03-18,2025-03-12,248,198,Delivered,NaN
80,PO/2023/004029,Malabar Spices & Foods,SKU-10173,2023-04-04,2023-04-27,2020-01-01,2023-04-04,2023-04-27,2020-01-01,506,455,Delivered,NaN
93,PO/2026/078127,Rajdhani Wholesale Mart,SKU-10229,2026-04-13,28.04.2026,05/01/2026,2026-04-13,2026-04-28,2026-01-05,69,69,Delivered,NaN
125,PO/2025/046663,Deccan Traders,SKU-10693,04/11/2025,2025-04-30,2025-04-30,2025-11-04,2025-04-30,2025-04-30,39,35,Delivered,Diwali rush
215,PO/2025/041349,Gujarat Chemicals Trading,SKU-10452,2025-12-04,10.12.2025,12/10/2025,2025-12-04,2025-12-10,2025-10-12,48,<NA>,Delivered,??
233,PO/2024/048041,Om Sai Trading Co,SKU-10593,2024-08-20,2024-08-29,2020-01-01,2024-08-20,2024-08-29,2020-01-01,365,357,Delivered,??


=== Cancelled orders with receipt dates ===


,order_id,supplier,sku,order_date,promised_date,received_date,order_date_parsed,promised_date_parsed,received_date_parsed,order_qty,received_qty,status,notes
18,PO/2024/093273,Reliable Industrial Corp,SKU-10700,06/03/2024,2024-03-25,2024-03-15,2024-03-06,2024-03-25,2024-03-15,699,699,Cancelled,monsoon delay - NH blocked
156,PO/2023/029471,Sri Lakshmi Distributors,SKU-10757,2023-06-20,2023-07-11,20/07/2023,2023-06-20,2023-07-11,2023-07-20,75,73,Cancelled,NaN
250,PO/2026/025158,Kisan Agri Hub,SKU-10149,2026-06-29,2026-07-17,26/07/2026,2026-06-29,2026-07-17,2026-07-26,-5,93,Cancelled,NaN
602,PO/2023/077802,Punjab Industrial Supplies,SKU-10434,2023-09-03,09/09/2023,2023-09-07,2023-09-03,2023-09-09,2023-09-07,149,149,Cancelled,Diwali rush
617,PO/2026/010461,Venkateswara Hardware,SKU-10288,2026-03-31,2026-04-11,2026-04-18,2026-03-31,2026-04-11,2026-04-18,121,114,Cancelled,??
719,PO/2025/095181,Punjab Industrial Supplies,SKU-10774,2025-12-05,2025-12-18,11-Dec-25,2025-12-05,2025-12-18,2025-12-11,171,171,Cancelled,short shipment
735,PO/2026/031841,Reliable Industrial Corp,SKU-10235,2026-08-23,2026-09-16,2026-09-04,2026-08-23,2026-09-16,2026-09-04,42,41,Cancelled,NaN
799,PO/2023/044401,Himalaya Wholesale,SKU-10312,27-10-2023,2023-11-07,05-11-2023,2023-10-27,2023-11-07,2023-11-05,109,109,Cancelled,recheck count
808,PO/2023/032058,PUNJAB INDUSTRIAL SUPPLIES,SKU-10038,23.11.2023,2023-12-15,2020-01-01,2023-11-23,2023-12-15,2020-01-01,71,71,Cancelled,e-way bill pending
839,PO/2023/027119,Maruti Logistics Pvt Ltd,SKU-10337,05/12/2023,03.06.2023,2023-05-22,2023-12-05,2023-06-03,2023-05-22,844,844,Cancelled,monsoon delay - NH blocked


=== Suspected default receipt dates ===


,order_id,supplier,sku,order_date,promised_date,received_date,order_date_parsed,promised_date_parsed,received_date_parsed,order_qty,received_qty,status,notes
1,PO/2024/034209,Himalaya Wholesale,SKU-10170,2024-10-27,2024-11-19,2020-01-01,2024-10-27,2024-11-19,2020-01-01,369,295,Delivered,NaN
14,PO/2026/054586,Surya Electricals,SKU-10270,2026-01-01,01/18/2026,2020-01-01,2026-01-01,2026-01-18,2020-01-01,198,178,Delivered,NaN
80,PO/2023/004029,Malabar Spices & Foods,SKU-10173,2023-04-04,2023-04-27,2020-01-01,2023-04-04,2023-04-27,2020-01-01,506,455,Delivered,NaN
233,PO/2024/048041,Om Sai Trading Co,SKU-10593,2024-08-20,2024-08-29,2020-01-01,2024-08-20,2024-08-29,2020-01-01,365,357,Delivered,??
312,PO/2024/084478,Tirupati Steel Works,SKU-10714,2024-04-14,2024-05-02,2020-01-01,2024-04-14,2024-05-02,2020-01-01,236,236,Delivered,damaged in transit
371,PO/2025/005812,Sai Krupa Electronics,SKU-10220,2025-05-29,2025-06-05,2020-01-01,2025-05-29,2025-06-05,2020-01-01,142,<NA>,Delivered,monsoon delay - NH blocked
413,PO/2023/032416,Himalaya Wholesale,SKU-10017,2023-04-16,04/20/2023,2020-01-01,2023-04-16,2023-04-20,2020-01-01,566,566,Delivered,urgent
503,PO/2025/071176,Coastal Cargo Suppliers,SKU-10041,11/24/2025,11/12/2025,2020-01-01,2025-11-24,2025-12-11,2020-01-01,135,135,Delivered,urgent
600,PO/2026/074551,Coastal Cargo Suppliers,SKU-10186,07/08/2026,2026-07-24,2020-01-01,2026-08-07,2026-07-24,2020-01-01,247,242,Delivered,??
626,PO/2023/095631,Ambika Pharma Distributors,SKU-10569,28-May-23,11-Jun-23,2020-01-01,2023-05-28,2023-06-11,2020-01-01,74,70,Delivered,LR copy missing


In [36]:
df_dates_final = df_dates_clean.copy()

# Identify the suspected default receipt date
default_date_mask = (
    df_dates_final["received_date_parsed"]
    .eq(pd.Timestamp("2020-01-01"))
    .fillna(False)
)

# Preserve an audit flag
df_dates_final["flag_suspected_default_received_date"] = default_date_mask

# Replace only the parsed date, not the original raw value
df_dates_final.loc[
    default_date_mask,
    "received_date_parsed"
] = pd.NaT

print("Suspected default dates flagged:", default_date_mask.sum())
print(
    "Remaining parsed receipt dates equal to 2020-01-01:",
    df_dates_final["received_date_parsed"]
    .eq(pd.Timestamp("2020-01-01"))
    .sum()
)

print("Dataset shape:", df_dates_final.shape)

Suspected default dates flagged: 1500
Remaining parsed receipt dates equal to 2020-01-01: 0
Dataset shape: (100800, 44)


In [37]:
# Step 15: Summarize date conflicts by status

conflict_summary = (
    df_dates_final
    .groupby("status", dropna=False)
    .agg(
        total_orders=("order_id", "size"),
        promised_before_order=("flag_promised_before_order", "sum"),
        received_before_order=("flag_received_before_order", "sum"),
        cancelled_with_receipt=("flag_cancelled_with_receipt_date", "sum")
    )
    .sort_values("total_orders", ascending=False)
)

print(conflict_summary)

                    total_orders  promised_before_order  \
status                                                    
Delivered                  77984                   2082   
In Transit                  6902                    190   
Partially Received          5992                    164   
Cancelled                   4995                    145   
Backordered                 4927                    130   

                    received_before_order  cancelled_with_receipt  
status                                                             
Delivered                            3160                       0  
In Transit                            118                       0  
Partially Received                    243                       0  
Cancelled                             109                    1539  
Backordered                            90                       0  


In [38]:
# Promised date earlier than order date
display(
    df_dates_final.loc[
        df_dates_final["flag_promised_before_order"],
        [
            "order_id", "order_date", "promised_date",
            "order_date_parsed", "promised_date_parsed",
            "status", "notes"
        ]
    ].head(15)
)

# Received date earlier than order date
display(
    df_dates_final.loc[
        df_dates_final["flag_received_before_order"],
        [
            "order_id", "order_date", "received_date",
            "order_date_parsed", "received_date_parsed",
            "status", "notes"
        ]
    ].head(15)
)

,order_id,order_date,promised_date,order_date_parsed,promised_date_parsed,status,notes
22,PO/2025/028311,2025-11-12,12/06/2025,2025-11-12,2025-06-12,Backordered,TBD
40,PO/2024/027704,10/03/2024,04/03/2024,2024-03-10,2024-03-04,Backordered,recheck count
44,PO/2026/081063,02/05/2026,20-Feb-26,2026-05-02,2026-02-20,Cancelled,LR copy missing
46,PO/2024/085116,07/08/2024,2024-07-26,2024-08-07,2024-07-26,Delivered,urgent
51,PO/2025/087197,03/07/2025,2025-03-18,2025-07-03,2025-03-18,Delivered,NaN
97,PO/2025/094683,16/07/2025,08/06/2025,2025-07-16,2025-06-08,Delivered,NaN
125,PO/2025/046663,04/11/2025,2025-04-30,2025-11-04,2025-04-30,Delivered,Diwali rush
265,PO/2026/032155,2026-07-29,08/02/2026,2026-07-29,2026-02-08,In Transit,NaN
313,PO/2024/022539,2024-08-01,08/05/2024,2024-08-01,2024-05-08,Delivered,monsoon delay - NH blocked
353,PO/2023/078883,2023-11-23,12/07/2023,2023-11-23,2023-07-12,Backordered,NaN


,order_id,order_date,received_date,order_date_parsed,received_date_parsed,status,notes
1,PO/2024/034209,2024-10-27,2020-01-01,2024-10-27,NaT,Delivered,NaN
14,PO/2026/054586,2026-01-01,2020-01-01,2026-01-01,NaT,Delivered,NaN
19,PO/2026/035745,2026-07-21,08/04/2026,2026-07-21,2026-04-08,Delivered,NaN
46,PO/2024/085116,07/08/2024,2024-08-03,2024-08-07,2024-08-03,Delivered,urgent
51,PO/2025/087197,03/07/2025,2025-03-12,2025-07-03,2025-03-12,Delivered,NaN
80,PO/2023/004029,2023-04-04,2020-01-01,2023-04-04,NaT,Delivered,NaN
93,PO/2026/078127,2026-04-13,05/01/2026,2026-04-13,2026-01-05,Delivered,NaN
125,PO/2025/046663,04/11/2025,2025-04-30,2025-11-04,2025-04-30,Delivered,Diwali rush
215,PO/2025/041349,2025-12-04,12/10/2025,2025-12-04,2025-10-12,Delivered,??
233,PO/2024/048041,2024-08-20,2020-01-01,2024-08-20,NaT,Delivered,??


In [39]:
import pandas as pd

df_date_review = df_dates_final.copy()

raw_date_cols = [
    "order_date",
    "promised_date",
    "received_date"
]

for col in raw_date_cols:
    raw = df_date_review[col].astype("string").str.strip()

    # Match dates like 05/08/2025, where both components
    # could be interpreted as a day or a month
    ambiguous = raw.str.match(
        r"^\d{1,2}/\d{1,2}/\d{4}$",
        na=False
    )

    parts = raw.str.extract(
        r"^(\d{1,2})/(\d{1,2})/(\d{4})$"
    )

    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")

    # Both day-first and month-first interpretations are valid
    truly_ambiguous = (
        ambiguous
        & first.between(1, 12)
        & second.between(1, 12)
        & first.ne(second)
    )

    print(f"\n{col}")
    print("Slash-formatted dates:", ambiguous.sum())
    print("Potentially ambiguous dates:", truly_ambiguous.sum())

    print("Examples:")
    print(raw[truly_ambiguous].head(10).to_list())


order_date
Slash-formatted dates: 15132
Potentially ambiguous dates: 5442
Examples:
['06/03/2024', '10/03/2024', '02/05/2026', '07/08/2024', '03/07/2025', '09/08/2026', '07/12/2023', '06/01/2023', '12/02/2024', '04/11/2025']

promised_date
Slash-formatted dates: 15919
Potentially ambiguous dates: 5399
Examples:
['09/04/2025', '12/06/2025', '04/07/2025', '02/05/2024', '04/03/2024', '03/07/2026', '04/12/2023', '05/02/2025', '05/06/2023', '08/06/2025']

received_date
Slash-formatted dates: 12919
Potentially ambiguous dates: 4693
Examples:
['10/01/2024', '08/04/2026', '02/08/2025', '01/05/2026', '05/01/2026', '02/03/2025', '08/09/2023', '06/09/2024', '12/10/2025', '10/08/2023']


In [40]:
import pandas as pd

df_date_review = df_dates_final.copy()

date_cols = ["order_date", "promised_date", "received_date"]

for col in date_cols:
    raw = df_date_review[col].astype("string").str.strip()

    # Restrict comparison to slash-formatted dates
    mask = raw.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)

    day_first = pd.to_datetime(
        raw.where(mask),
        format="%d/%m/%Y",
        errors="coerce"
    )

    month_first = pd.to_datetime(
        raw.where(mask),
        format="%m/%d/%Y",
        errors="coerce"
    )

    # Dates valid under only one interpretation
    day_only = day_first.notna() & month_first.isna()
    month_only = month_first.notna() & day_first.isna()
    both_valid = day_first.notna() & month_first.notna()

    print(f"\n--- {col} ---")
    print("Slash-formatted dates:", int(mask.sum()))
    print("Valid day-first only:", int(day_only.sum()))
    print("Valid month-first only:", int(month_only.sum()))
    print("Valid under both:", int(both_valid.sum()))
    print("Invalid under both:", int((mask & day_first.isna() & month_first.isna()).sum()))

    print("\nDay-first only examples:")
    print(raw[day_only].head(10).to_list())

    print("\nMonth-first only examples:")
    print(raw[month_only].head(10).to_list())


--- order_date ---
Slash-formatted dates: 15132
Valid day-first only: 4612
Valid month-first only: 4580
Valid under both: 5940
Invalid under both: 0

Day-first only examples:
['20/01/2025', '17/04/2026', '15/06/2023', '28/02/2025', '18/10/2025', '13/02/2026', '27/11/2023', '20/04/2023', '23/01/2026', '16/07/2025']

Month-first only examples:
['05/22/2024', '11/13/2024', '12/23/2023', '07/24/2024', '01/24/2024', '04/25/2023', '02/20/2025', '10/14/2024', '05/30/2025', '11/20/2025']

--- promised_date ---
Slash-formatted dates: 15919
Valid day-first only: 4497
Valid month-first only: 4560
Valid under both: 5879
Invalid under both: 983

Day-first only examples:
['15/09/2023', '27/08/2024', '26/05/2024', '23/12/2023', '25/03/2026', '21/04/2026', '31/12/2025', '28/08/2026', '19/11/2025', '19/06/2024']

Month-first only examples:
['01/18/2026', '01/19/2025', '07/17/2023', '07/26/2025', '01/20/2025', '02/25/2024', '12/18/2024', '02/20/2026', '10/22/2023', '06/24/2026']

--- received_date ---


In [41]:
import pandas as pd

df_date_review = df_dates_final.copy()

date_cols = ["order_date", "promised_date", "received_date"]

for col in date_cols:
    raw = df_date_review[col].astype("string").str.strip()

    mask = raw.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)

    day_first = pd.to_datetime(
        raw.where(mask),
        format="%d/%m/%Y",
        errors="coerce"
    )

    month_first = pd.to_datetime(
        raw.where(mask),
        format="%m/%d/%Y",
        errors="coerce"
    )

    day_only = day_first.notna() & month_first.isna()
    month_only = month_first.notna() & day_first.isna()
    both_valid = day_first.notna() & month_first.notna()
    neither_valid = mask & day_first.isna() & month_first.isna()

    print(f"\n--- {col} ---")
    print("Slash-formatted:", int(mask.sum()))
    print("Day-first only:", int(day_only.sum()))
    print("Month-first only:", int(month_only.sum()))
    print("Valid under both:", int(both_valid.sum()))
    print("Invalid under both:", int(neither_valid.sum()))

    print("\nDay-first only examples:")
    print(raw[day_only].head(10).to_list())

    print("\nMonth-first only examples:")
    print(raw[month_only].head(10).to_list())


--- order_date ---
Slash-formatted: 15132
Day-first only: 4612
Month-first only: 4580
Valid under both: 5940
Invalid under both: 0

Day-first only examples:
['20/01/2025', '17/04/2026', '15/06/2023', '28/02/2025', '18/10/2025', '13/02/2026', '27/11/2023', '20/04/2023', '23/01/2026', '16/07/2025']

Month-first only examples:
['05/22/2024', '11/13/2024', '12/23/2023', '07/24/2024', '01/24/2024', '04/25/2023', '02/20/2025', '10/14/2024', '05/30/2025', '11/20/2025']

--- promised_date ---
Slash-formatted: 15919
Day-first only: 4497
Month-first only: 4560
Valid under both: 5879
Invalid under both: 983

Day-first only examples:
['15/09/2023', '27/08/2024', '26/05/2024', '23/12/2023', '25/03/2026', '21/04/2026', '31/12/2025', '28/08/2026', '19/11/2025', '19/06/2024']

Month-first only examples:
['01/18/2026', '01/19/2025', '07/17/2023', '07/26/2025', '01/20/2025', '02/25/2024', '12/18/2024', '02/20/2026', '10/22/2023', '06/24/2026']

--- received_date ---
Slash-formatted: 12919
Day-first onl

In [42]:
import pandas as pd

df_date_review = df_dates_final.copy()

date_cols = ["order_date", "promised_date", "received_date"]

# Create a reliable reference: ambiguous slash dates are excluded
for col in date_cols:
    raw = df_date_review[col].astype("string").str.strip()

    slash = raw.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)
    parts = raw.str.extract(r"^(\d{1,2})/(\d{1,2})/(\d{4})$")

    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")

    day_first = pd.to_datetime(
        raw.where(slash), format="%d/%m/%Y", errors="coerce"
    )
    month_first = pd.to_datetime(
        raw.where(slash), format="%m/%d/%Y", errors="coerce"
    )

    ambiguous = (
        slash
        & day_first.notna()
        & month_first.notna()
        & first.ne(second)
    )

    # Start with existing parsed dates
    df_date_review[f"{col}_reliable"] = (
        df_date_review[f"{col}_parsed"].copy()
    )

    # Exclude ambiguous slash dates from reference comparisons
    df_date_review.loc[ambiguous, f"{col}_reliable"] = pd.NaT

    # Correctly parse slash dates valid under only one interpretation
    day_only = slash & day_first.notna() & month_first.isna()
    month_only = slash & month_first.notna() & day_first.isna()

    df_date_review.loc[day_only, f"{col}_reliable"] = day_first[day_only]
    df_date_review.loc[month_only, f"{col}_reliable"] = month_first[month_only]


# Test ambiguous dates against reliable dates in their rows
results = []

for col in date_cols:
    raw = df_date_review[col].astype("string").str.strip()
    slash = raw.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False)

    parts = raw.str.extract(r"^(\d{1,2})/(\d{1,2})/(\d{4})$")
    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")

    day_first = pd.to_datetime(
        raw.where(slash), format="%d/%m/%Y", errors="coerce"
    )
    month_first = pd.to_datetime(
        raw.where(slash), format="%m/%d/%Y", errors="coerce"
    )

    ambiguous = (
        slash
        & day_first.notna()
        & month_first.notna()
        & first.ne(second)
    )

    for idx in df_date_review.index[ambiguous]:
        order_date = df_date_review.at[idx, "order_date_reliable"]
        promised_date = df_date_review.at[idx, "promised_date_reliable"]
        received_date = df_date_review.at[idx, "received_date_reliable"]

        candidates = {
            "day-first": day_first.at[idx],
            "month-first": month_first.at[idx]
        }

        scores = {}

        for fmt, candidate in candidates.items():
            order = candidate if col == "order_date" else order_date
            promised = candidate if col == "promised_date" else promised_date
            received = candidate if col == "received_date" else received_date

            # Count timeline violations only where both dates are reliable
            score = 0

            if pd.notna(order) and pd.notna(promised) and order > promised:
                score += 1

            if pd.notna(order) and pd.notna(received) and order > received:
                score += 1

            scores[fmt] = score

        if scores["day-first"] < scores["month-first"]:
            assessment = "Day-first fits better"
        elif scores["month-first"] < scores["day-first"]:
            assessment = "Month-first fits better"
        else:
            assessment = "Unresolved"

        results.append({
            "row_index": idx,
            "order_id": df_date_review.at[idx, "order_id"],
            "column": col,
            "raw_date": raw.at[idx],
            "day_first_candidate": candidates["day-first"],
            "month_first_candidate": candidates["month-first"],
            "day_first_violations": scores["day-first"],
            "month_first_violations": scores["month-first"],
            "assessment": assessment
        })

df_ambiguous_review = pd.DataFrame(results)

print("Ambiguous dates reviewed:", len(df_ambiguous_review))
print("\nAssessment counts:")
print(df_ambiguous_review["assessment"].value_counts())

print("\nExamples where day-first fits better:")
print(
    df_ambiguous_review[
        df_ambiguous_review["assessment"] == "Day-first fits better"
    ].head(10).to_string(index=False)
)

print("\nExamples where month-first fits better:")
print(
    df_ambiguous_review[
        df_ambiguous_review["assessment"] == "Month-first fits better"
    ].head(10).to_string(index=False)
)

Ambiguous dates reviewed: 15534

Assessment counts:
assessment
Unresolved                 7944
Month-first fits better    3803
Day-first fits better      3787
Name: count, dtype: int64

Examples where day-first fits better:
 row_index       order_id     column   raw_date day_first_candidate month_first_candidate  day_first_violations  month_first_violations            assessment
        18 PO/2024/093273 order_date 06/03/2024          2024-03-06            2024-06-03                     0                       2 Day-first fits better
        53 PO/2026/054998 order_date 09/08/2026          2026-08-09            2026-09-08                     0                       2 Day-first fits better
       112 PO/2024/067255 order_date 12/02/2024          2024-02-12            2024-12-02                     0                       2 Day-first fits better
       136 PO/2025/074791 order_date 06/04/2025          2025-04-06            2025-06-04                     0                       2 Day-firs

In [43]:
# Save current dataset and date review to Excel

output_file = "Supply_Chain_Inventory_Date_Review.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_dates_final.to_excel(
        writer,
        sheet_name="Current_Cleaned_Data",
        index=False
    )

    df_ambiguous_review.to_excel(
        writer,
        sheet_name="Ambiguous_Date_Review",
        index=False
    )

print(f"File saved successfully: {output_file}")
print(f"Dataset rows: {len(df_dates_final):,}")
print(f"Ambiguous dates for review: {len(df_ambiguous_review):,}")

File saved successfully: Supply_Chain_Inventory_Date_Review.xlsx
Dataset rows: 100,800
Ambiguous dates for review: 15,534


In [44]:
output_file = "Supply_Chain_Inventory_Cleaned.csv"

df_dates_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"File saved successfully: {output_file}")
print(f"Rows: {len(df_dates_final):,}")
print(f"Columns: {len(df_dates_final.columns)}")

File saved successfully: Supply_Chain_Inventory_Cleaned.csv
Rows: 100,800
Columns: 44
